# Liu2024 S-JEPA Embeddings + Shrinkage LDA

**Goal:** Test whether frozen S-JEPA PreLocal embeddings are useful when we replace the unstable
neural classification head with a small classical classifier (shrinkage LDA or L2-logistic).

**Key difference from the existing S-JEPA notebook:**
- No braindecode EEGClassifier / skorch training loop.
- The S-JEPA local encoder is frozen. Only the spatial_conv is fine-tuned per fold (same as `new` strategy).
- After extracting per-trial embeddings, we train shrinkage LDA or logistic regression.
- With 24 training trials and potentially 100s of embedding dimensions, PCA is applied inside the training fold.

**Leakage controls:**
- Preprocessing (resample, bandpass, average reference) uses fixed transforms — safe.
- Spatial_conv fine-tuning uses only training fold trials.
- PCA is fit on training fold embeddings only.
- Classifier is fit on training fold features only.
- Nothing from the test fold is ever seen during fitting.

## 1. Imports

In [14]:
import os
import re
import sys
import json
import random
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt

import torch
import torch.nn as nn
import torch.optim as optim

from scipy.io import loadmat
from scipy import signal as sp_signal

import mne
mne.set_log_level("WARNING")

from sklearn.model_selection import StratifiedShuffleSplit
from sklearn.discriminant_analysis import LinearDiscriminantAnalysis
from sklearn.linear_model import LogisticRegression
from sklearn.decomposition import PCA
from sklearn.metrics import accuracy_score, balanced_accuracy_score, confusion_matrix

from braindecode.models import SignalJEPA_PreLocal

warnings.filterwarnings('ignore', category=RuntimeWarning)
print(f"torch {torch.__version__}, numpy {np.__version__}")

def resolve_device(cfg_device="auto"):
    if cfg_device != "auto":
        return torch.device(cfg_device)
    if torch.backends.mps.is_available() and torch.backends.mps.is_built():
        return torch.device("mps")
    if torch.cuda.is_available():
        return torch.device("cuda")
    return torch.device("cpu")

torch 2.10.0+cu128, numpy 2.4.3


## 2. CONFIG

In [15]:
CONFIG = {
    # --- Paths ---
    "data_root":             "../../liu2024_data/liu2024_figshare/sourcedata",
    "artifact_root":         "../../artifacts/liu2024_sjepa_embeddings_lda",
    # S-JEPA checkpoint: set to None to use HuggingFace from_pretrained
    "sjepa_checkpoint_path": None,
    "sjepa_repo_id":         "braindecode/signal-jepa_without-chans",

    # --- Dataset ---
    "subjects":   "all",
    "random_state": 2026,
    "sfreq_raw":  500,
    "sfreq_model": 128,
    # MI cue (marker==2) sits at ~2.0 s in the trial; start at 1.5 s so the 4.2 s window is
    # onset-aligned (1.5-5.7 s) instead of the pre-imagery period. Matches the augmented notebook.
    "mi_window_s": (1.5, 5.7),
    "bandpass_hz": (0.5, 40.0),

    # --- S-JEPA window (must match what the model was trained with) ---
    # S-JEPA PreLocal expects input of this many samples at sfreq_model Hz.
    # At 128 Hz, 4.195 s ≈ 537 samples. Use same as existing notebook.
    "window_samples": 537,

    # --- Cross-validation ---
    "n_repeats":  10,
    "test_size":  0.40,

    # --- Embedding ---
    # Rich embedding via forward hook (resolved limitation, see markdown below).
    "embedding_hook": "feature_encoder",  # 'feature_encoder' (rich tokens) or 'spatial_conv'
    "embedding_pool": "mean",  # 'mean'|'max'|'meanmax' over tokens, or 'flatten'
    "use_pca":        True,
    "pca_max_components": 20,  # with 24 train trials, keep well below 24

    # --- Classifier ---
    # 'shrinkage_lda' or 'logistic_l2'
    "classifier": "logistic_l2",
    "logistic_C":  1.0,           # used only when classifier=logistic_l2

    # --- Spatial conv fine-tuning ---
    # Mirrors the 'new' strategy: fine-tune only spatial_conv + final_layer
    # on the training fold before embedding extraction.
    "finetune_spatial_conv": True,
    "finetune_epochs":       30,
    "finetune_lr":           1e-3,
    "finetune_batch_size":   8,
    "finetune_patience":     10,

    # --- Misc ---
    "save_embeddings": True,    # save raw embeddings to disk for later PRISM use
    "device": "auto",
}

DATA_ROOT    = Path(CONFIG["data_root"])
ARTIFACT_ROOT = Path(CONFIG["artifact_root"])
ARTIFACT_ROOT.mkdir(parents=True, exist_ok=True)

DEVICE = resolve_device(CONFIG["device"])
SFREQ_RAW   = CONFIG["sfreq_raw"]
SFREQ_MODEL = CONFIG["sfreq_model"]
MI_SAMPLES_RAW   = int((CONFIG["mi_window_s"][1] - CONFIG["mi_window_s"][0]) * SFREQ_RAW)
WINDOW_SAMPLES   = CONFIG["window_samples"]

np.random.seed(CONFIG["random_state"])
random.seed(CONFIG["random_state"])
torch.manual_seed(CONFIG["random_state"])

print(f"Device:       {DEVICE}")
print(f"Data root:    {DATA_ROOT}")
print(f"Artifacts:    {ARTIFACT_ROOT}")
print(f"MI window:    {CONFIG['mi_window_s']} s → {MI_SAMPLES_RAW} samples at {SFREQ_RAW} Hz")
print(f"Model input:  {WINDOW_SAMPLES} samples at {SFREQ_MODEL} Hz")
print(f"Classifier:   {CONFIG['classifier']}")

Device:       cpu
Data root:    ../../liu2024_data/liu2024_figshare/sourcedata
Artifacts:    ../../artifacts/liu2024_sjepa_embeddings_lda
MI window:    (1.5, 5.7) s → 2100 samples at 500 Hz
Model input:  537 samples at 128 Hz
Classifier:   logistic_l2


## 3. Liu2024 Channel Constants

In [16]:
SOURCE_EEG_NAMES_30 = [
    "Fp1", "Fp2", "Fz", "F3", "F4", "F7", "F8", "FCz", "FC3", "FC4",
    "FT7", "FT8", "Cz", "C3", "C4", "T3", "T4", "CPz",
    "CP3", "CP4", "TP7", "TP8", "Pz", "P3", "P4", "T5", "T6", "Oz", "O1", "O2",
]
CPZ_IDX     = 17
EEG_KEEP_IDX = [i for i in range(30) if i != CPZ_IDX]
EEG_NAMES    = [SOURCE_EEG_NAMES_30[i] for i in EEG_KEEP_IDX]
N_CHANS      = len(EEG_KEEP_IDX)  # 29

def make_liu_info(sfreq):
    info = mne.create_info(
        ch_names=EEG_NAMES,
        sfreq=float(sfreq),
        ch_types=["eeg"] * N_CHANS,
    )
    montage = mne.channels.make_standard_montage("standard_1020")
    info.set_montage(montage, match_case=False, on_missing="ignore")
    return info

MNE_INFO = make_liu_info(SFREQ_MODEL)
CHS_INFO = MNE_INFO["chs"]
print(f"N channels: {N_CHANS}, ch_names[:5]: {EEG_NAMES[:5]}")

N channels: 29, ch_names[:5]: ['Fp1', 'Fp2', 'Fz', 'F3', 'F4']


## 4. Data Loading and Preprocessing

In [17]:
def subject_id_from_path(path):
    m = re.search(r"sub[-_ ]?(\d{1,2})", str(path), flags=re.IGNORECASE)
    return int(m.group(1)) if m else int(re.findall(r"\d+", Path(path).stem)[-1])


def preprocess_subject(mat_path):
    """
    Load Liu2024 source .mat → (40, 29, WINDOW_SAMPLES) at SFREQ_MODEL Hz, y (40,).
    Uses MNE RawArray for average reference + resample + bandpass.
    Returns float32 array compatible with S-JEPA PreLocal.
    """
    mat = loadmat(str(mat_path), squeeze_me=True, struct_as_record=False)
    raw_data = mat.get("rawdata", mat.get("data", None))
    labels   = mat.get("labels", mat.get("label", None))
    # Liu2024 figshare files nest arrays under an 'eeg' struct: eeg.rawdata / eeg.label
    if raw_data is None or labels is None:
        for k, v in mat.items():
            if k.startswith("__"):
                continue
            if hasattr(v, "_fieldnames"):
                if raw_data is None and "rawdata" in v._fieldnames:
                    raw_data = getattr(v, "rawdata")
                if labels is None and "label" in v._fieldnames:
                    labels = getattr(v, "label")
            elif raw_data is None and isinstance(v, np.ndarray) and v.ndim == 3:
                raw_data = v

    raw_data = np.asarray(raw_data, dtype=np.float64)
    # Normalise to trials x 33 x 4000
    trial_ax = next(ax for ax, sz in enumerate(raw_data.shape) if sz == 40)
    raw_data = np.moveaxis(raw_data, trial_ax, 0)
    if raw_data.shape[2] == 33:
        raw_data = raw_data.transpose(0, 2, 1)  # wait — channels must be dim 1
    # After moveaxis, shape should be 40 x 33 x 4000 OR 40 x 4000 x 33
    if raw_data.shape[1] != 33 and raw_data.shape[2] == 33:
        raw_data = raw_data.transpose(0, 2, 1)
    assert raw_data.shape == (40, 33, 4000), f"Shape mismatch: {raw_data.shape}"

    y = np.asarray(labels, dtype=int).ravel()
    if set(np.unique(y).tolist()).issubset({1, 2}):
        y = y - 1

    # Select 29 EEG channels
    eeg = raw_data[:, EEG_KEEP_IDX, :].astype(np.float64)  # 40 x 29 x 4000

    # Convert to MNE RawArray per subject (concatenate trials as continuous signal)
    # This gives us average reference and proper FIR filtering
    n_trials, n_ch, n_t = eeg.shape
    continuous = eeg.reshape(n_ch, n_trials * n_t) * 1e-6  # → Volts for MNE
    info_raw = mne.create_info(ch_names=EEG_NAMES, sfreq=float(SFREQ_RAW), ch_types=["eeg"] * N_CHANS)
    raw_mne = mne.io.RawArray(continuous, info_raw, verbose=False)

    # Average reference
    raw_mne.set_eeg_reference("average", projection=False, verbose=False)
    # Resample to 128 Hz
    raw_mne.resample(SFREQ_MODEL, npad="auto", verbose=False)
    # Bandpass 0.5–40 Hz
    bp_low, bp_high = CONFIG["bandpass_hz"]
    raw_mne.filter(bp_low, bp_high, method="fir", phase="zero", verbose=False)

    # Reshape back to trials
    data_resampled = raw_mne.get_data() * 1e6  # back to microvolts
    n_t_resampled = int(n_t * SFREQ_MODEL / SFREQ_RAW)
    data_trials = data_resampled.reshape(N_CHANS, n_trials, n_t_resampled)
    data_trials = data_trials.transpose(1, 0, 2)  # → 40 x 29 x n_t_resampled

    # Extract MI window and crop to WINDOW_SAMPLES
    mi_start_s = CONFIG["mi_window_s"][0]
    mi_start_idx = int(mi_start_s * SFREQ_MODEL)
    mi_stop_idx  = mi_start_idx + WINDOW_SAMPLES
    assert mi_stop_idx <= n_t_resampled, (
        f"Window [{mi_start_idx}:{mi_stop_idx}] exceeds resampled length {n_t_resampled}"
    )
    X = data_trials[:, :, mi_start_idx:mi_stop_idx].astype(np.float32)  # 40 x 29 x 537

    assert X.shape == (40, N_CHANS, WINDOW_SAMPLES), f"Final X shape mismatch: {X.shape}"
    assert np.isfinite(X).all(), "Non-finite values in preprocessed data"
    return X, y


def find_mat_files(root):
    root = Path(root)
    if not root.exists():
        raise FileNotFoundError(f"Data root not found: {root}")
    return sorted(root.rglob("*.mat"))


mat_files  = find_mat_files(DATA_ROOT)
all_sids   = sorted({subject_id_from_path(f) for f in mat_files})
SUBJECT_IDS = all_sids if CONFIG["subjects"] == "all" else sorted(int(s) for s in CONFIG["subjects"])
sid_to_path = {subject_id_from_path(f): f for f in mat_files if subject_id_from_path(f) in SUBJECT_IDS}

print(f"Found {len(mat_files)} .mat files, using subjects: {SUBJECT_IDS[:5]}...")

Found 50 .mat files, using subjects: [1, 2, 3, 4, 5]...


## 5. S-JEPA Model Loading

In [18]:
NEW_LAYER_PREFIXES = ("spatial_conv.", "final_layer.")


def load_sjepa_model(cfg, n_chans, chs_info, n_times, n_outputs=2):
    """
    Load SignalJEPA_PreLocal from HuggingFace or a local checkpoint.
    Returns the model with all weights frozen except spatial_conv + final_layer.
    """
    kwargs = dict(
        n_chans=n_chans,
        chs_info=chs_info,
        n_times=n_times,
        n_outputs=n_outputs,
    )
    ckpt = cfg.get("sjepa_checkpoint_path")
    if ckpt is not None:
        print(f"Loading S-JEPA from local checkpoint: {ckpt}")
        model = SignalJEPA_PreLocal(**kwargs)
        state = torch.load(ckpt, map_location="cpu")
        missing, unexpected = model.load_state_dict(state, strict=False)
        print(f"  Missing keys: {missing[:5]}{'...' if len(missing) > 5 else ''}")
        print(f"  Unexpected keys: {unexpected[:5]}{'...' if len(unexpected) > 5 else ''}")
    else:
        print(f"Loading S-JEPA from HuggingFace: {cfg['sjepa_repo_id']}")
        model = SignalJEPA_PreLocal.from_pretrained(
            cfg["sjepa_repo_id"], **kwargs, strict=False
        )

    # Freeze everything
    for p in model.parameters():
        p.requires_grad = False

    # Unfreeze spatial_conv and final_layer
    for name, p in model.named_parameters():
        if any(name.startswith(prefix) for prefix in NEW_LAYER_PREFIXES):
            p.requires_grad = True

    total      = sum(p.numel() for p in model.parameters())
    trainable  = sum(p.numel() for p in model.parameters() if p.requires_grad)
    print(f"  Total params:     {total:,}")
    print(f"  Trainable params: {trainable:,} (spatial_conv + final_layer)")
    return model


# Load once; we'll reset the trainable weights per fold
BASE_MODEL = load_sjepa_model(CONFIG, N_CHANS, CHS_INFO, WINDOW_SAMPLES)
BASE_MODEL = BASE_MODEL.to(DEVICE)
print(f"Model on: {next(BASE_MODEL.parameters()).device}")

Loading S-JEPA from HuggingFace: braindecode/signal-jepa_without-chans
  Total params:     16,010
  Trainable params: 2,170 (spatial_conv + final_layer)
Model on: cpu


## 6. Spatial Conv Fine-tuning + Embedding Extraction

For each fold we:
1. Deep-copy the base model and reset spatial_conv + final_layer weights.
2. Fine-tune spatial_conv + final_layer on training trials (cross-entropy, early stopping on a held-out 20%).
3. Extract embeddings from the trained model for all train + test trials.

The embedding is the mean-pooled token sequence after the local encoder and spatial aggregation,
**before** the final classification layer. This gives a compact representation per trial.

In [19]:
import copy


class TrialDataset(torch.utils.data.Dataset):
    def __init__(self, X, y):
        self.X = torch.from_numpy(np.asarray(X, dtype=np.float32))
        self.y = torch.from_numpy(np.asarray(y, dtype=np.int64))
    def __len__(self):
        return len(self.y)
    def __getitem__(self, idx):
        return self.X[idx], self.y[idx]


def finetune_spatial_conv(model, X_train, y_train, cfg, device):
    """
    Fine-tune spatial_conv + final_layer on training trials.
    Uses 80/20 internal split for early stopping.
    Returns the fine-tuned model.
    """
    model = copy.deepcopy(model)
    # Re-init trainable weights to avoid carry-over between subjects
    for name, module in model.named_modules():
        if any(name.startswith(pf.rstrip(".")) for pf in NEW_LAYER_PREFIXES):
            if hasattr(module, "reset_parameters"):
                module.reset_parameters()

    model.train()
    optimizer = optim.Adam(
        [p for p in model.parameters() if p.requires_grad],
        lr=cfg["finetune_lr"],
        weight_decay=1e-4,
    )
    criterion = nn.CrossEntropyLoss()

    # Internal 80/20 split for early stopping
    n = len(X_train)
    n_val = max(2, int(0.2 * n))
    idx = np.random.permutation(n)
    val_idx, tr_idx = idx[:n_val], idx[n_val:]

    ds_tr  = TrialDataset(X_train[tr_idx],  y_train[tr_idx])
    ds_val = TrialDataset(X_train[val_idx], y_train[val_idx])
    dl_tr  = torch.utils.data.DataLoader(ds_tr,  batch_size=cfg["finetune_batch_size"], shuffle=True)
    dl_val = torch.utils.data.DataLoader(ds_val, batch_size=cfg["finetune_batch_size"], shuffle=False)

    best_val_loss = float("inf")
    best_state    = copy.deepcopy({k: v.cpu() for k, v in model.state_dict().items()})
    patience_ctr  = 0

    for epoch in range(cfg["finetune_epochs"]):
        model.train()
        for xb, yb in dl_tr:
            xb, yb = xb.to(device), yb.to(device)
            optimizer.zero_grad()
            out = model(xb)
            loss = criterion(out, yb)
            loss.backward()
            optimizer.step()

        model.eval()
        val_loss = 0.0
        with torch.no_grad():
            for xb, yb in dl_val:
                xb, yb = xb.to(device), yb.to(device)
                out = model(xb)
                val_loss += criterion(out, yb).item()
        val_loss /= max(len(dl_val), 1)

        if val_loss < best_val_loss - 1e-4:
            best_val_loss = val_loss
            best_state    = copy.deepcopy({k: v.cpu() for k, v in model.state_dict().items()})
            patience_ctr  = 0
        else:
            patience_ctr += 1
            if patience_ctr >= cfg["finetune_patience"]:
                break

    model.load_state_dict({k: v.to(device) for k, v in best_state.items()})
    model.eval()
    return model


@torch.no_grad()
def extract_embeddings(model, X, cfg, device):
    """
    Extract per-trial RICH embeddings from S-JEPA PreLocal via a forward hook.

    PreLocal forward is: spatial_conv -> feature_encoder -> final_layer (2-D head).
    We hook an intermediate module and pool to a fixed (n_trials, D) matrix:
      cfg['embedding_hook']:
        'feature_encoder' (default): rich local-token tensor (B, n_tokens, emb_dim) -> pool over tokens
        'spatial_conv'             : spatially-filtered signal (B, n_spat_filters, n_times) -> pool over time
      cfg['embedding_pool']:
        'mean' | 'max' | 'meanmax' over the pooled axis, or 'flatten' (full flatten).
    Returns: numpy (n_trials, D).  (D probed at runtime; no longer the 2-D logits.)
    """
    model.eval()
    pool = cfg.get("embedding_pool", "mean")
    hook_name = cfg.get("embedding_hook", "feature_encoder")
    target = getattr(model, hook_name, None)
    if target is None:
        raise AttributeError(f"model has no submodule '{hook_name}' to hook")

    captured = {}
    def _hook(module, inp, out):
        captured["z"] = out.detach()
    handle = target.register_forward_hook(_hook)

    ds = TrialDataset(X, np.zeros(len(X), dtype=np.int64))
    dl = torch.utils.data.DataLoader(ds, batch_size=32, shuffle=False)
    all_embs = []
    try:
        for xb, _ in dl:
            xb = xb.to(device)
            _ = model(xb)                      # full forward; hook captures the intermediate
            z = captured["z"]
            if z.dim() == 2:
                feat = z
            else:
                # feature_encoder -> pool over tokens (dim 1); spatial_conv -> pool over time (dim 2)
                axis = 2 if hook_name == "spatial_conv" else 1
                if pool == "flatten":
                    feat = z.flatten(start_dim=1)
                elif pool == "max":
                    feat = z.max(dim=axis).values.flatten(start_dim=1)
                elif pool == "meanmax":
                    feat = torch.cat([z.mean(dim=axis), z.max(dim=axis).values], dim=-1).flatten(start_dim=1)
                else:  # mean
                    feat = z.mean(dim=axis).flatten(start_dim=1)
            all_embs.append(feat.cpu().numpy())
    finally:
        handle.remove()

    embeddings = np.concatenate(all_embs, axis=0).astype(np.float32)
    if not hasattr(extract_embeddings, "_printed"):
        print(f"  [hook={hook_name} pool={pool}] embedding matrix: {embeddings.shape}")
        extract_embeddings._printed = True
    return embeddings


print("Fine-tuning and embedding extraction functions defined.")

Fine-tuning and embedding extraction functions defined.


> **Embedding dimensionality (resolved).** `extract_embeddings` now registers a forward hook on an
> intermediate S-JEPA module instead of using the 2-D class logits. With `embedding_hook="feature_encoder"`
> (default) it captures the rich local-token tensor `(batch, n_tokens, emb_dim=64)` and pools over the token
> axis (`embedding_pool` = `mean`/`max`/`meanmax`/`flatten`) to a fixed per-trial vector (D = 64 for `mean`,
> ~1024 for `flatten`). `embedding_hook="spatial_conv"` instead captures the spatially-filtered signal.
> This is the meaningful representation the LDA probe and the hybrid's S-JEPA branch need; the old 2-D-logit
> path is gone. Leakage is unchanged: spatial_conv fine-tune, PCA, and LDA are still fit on the train fold only.


## 7. Classifier

In [20]:
def make_clf(cfg):
    if cfg["classifier"] == "shrinkage_lda":
        return LinearDiscriminantAnalysis(solver="lsqr", shrinkage="auto")
    elif cfg["classifier"] == "logistic_l2":
        return LogisticRegression(C=cfg["logistic_C"], penalty="l2",
                                  solver="lbfgs", max_iter=500)
    else:
        raise ValueError(f"Unknown classifier: {cfg['classifier']}")


def collapse_diagnostics(y_pred, n_classes=2):
    counts = np.bincount(y_pred, minlength=n_classes)
    dominant = counts.max() / counts.sum() if counts.sum() > 0 else 1.0
    return {
        "collapse_flag":  bool(dominant > 0.95),
        "collapse_ratio": float(dominant),
        "pred_counts":    counts.tolist(),
    }


print("Classifier helpers defined.")

Classifier helpers defined.


## 8. Per-Subject Cross-Validation Runner

In [21]:
def run_subject(sid, X, y, cfg, base_model, device):
    sss = StratifiedShuffleSplit(
        n_splits=cfg["n_repeats"],
        test_size=cfg["test_size"],
        random_state=cfg["random_state"],
    )
    fold_results = []

    for fold_idx, (tr_idx, te_idx) in enumerate(sss.split(X, y)):
        X_train, X_test = X[tr_idx], X[te_idx]
        y_train, y_test = y[tr_idx], y[te_idx]

        assert len(np.unique(y_train)) == 2, f"Sub {sid} fold {fold_idx}: single class in train"
        assert np.isfinite(X_train).all() and np.isfinite(X_test).all()

        try:
            # --- Step 1: Fine-tune spatial_conv on training fold ---
            if cfg["finetune_spatial_conv"]:
                model = finetune_spatial_conv(base_model, X_train, y_train, cfg, device)
            else:
                model = copy.deepcopy(base_model)
                model.eval()

            # --- Step 2: Extract embeddings (fit on train, apply to both) ---
            emb_train = extract_embeddings(model, X_train, cfg, device)  # (n_train, D)
            emb_test  = extract_embeddings(model, X_test,  cfg, device)  # (n_test, D)

            # --- Step 3: PCA (fit on train only) ---
            if cfg["use_pca"] and emb_train.shape[1] > cfg["pca_max_components"]:
                n_comp = min(cfg["pca_max_components"], emb_train.shape[0] - 1)
                pca = PCA(n_components=n_comp, random_state=cfg["random_state"])
                emb_train = pca.fit_transform(emb_train)
                emb_test  = pca.transform(emb_test)

            # --- Step 4: Classify ---
            clf = make_clf(cfg)
            clf.fit(emb_train, y_train)
            y_pred = clf.predict(emb_test)

        except Exception as exc:
            print(f"  Sub {sid} fold {fold_idx} ERROR: {exc}")
            y_pred = np.zeros(len(y_test), dtype=int)

        acc  = float(accuracy_score(y_test, y_pred))
        bacc = float(balanced_accuracy_score(y_test, y_pred))
        cm   = confusion_matrix(y_test, y_pred, labels=[0, 1]).tolist()
        diag = collapse_diagnostics(y_pred)

        cm_arr = np.array(cm)
        left_recall  = cm_arr[0, 0] / cm_arr[0].sum() if cm_arr[0].sum() > 0 else float("nan")
        right_recall = cm_arr[1, 1] / cm_arr[1].sum() if cm_arr[1].sum() > 0 else float("nan")

        fold_results.append({
            "subject_id":        sid,
            "fold_id":           fold_idx,
            "accuracy":          acc,
            "balanced_accuracy": bacc,
            "left_recall":       float(left_recall),
            "right_recall":      float(right_recall),
            "confusion_matrix":  cm,
            "collapse_flag":     diag["collapse_flag"],
            "collapse_ratio":    diag["collapse_ratio"],
            "pred_counts":       diag["pred_counts"],
            "n_train":           int(len(y_train)),
            "n_test":            int(len(y_test)),
            "embedding_dim":     int(emb_train.shape[1]) if 'emb_train' in dir() else -1,
        })

    return fold_results


print("Subject runner defined.")

Subject runner defined.


## 9. Run All Subjects

In [22]:
ALL_FOLD_RESULTS  = []
SUBJECT_SUMMARIES = []
SAVED_EMBEDDINGS  = {}  # {sid: {"X_train": ..., "X_test": ..., "y": ...}} if save_embeddings

print(f"Running {len(SUBJECT_IDS)} subjects | clf={CONFIG['classifier']}")
print("=" * 60)

for sid in SUBJECT_IDS:
    mat_path = sid_to_path.get(sid)
    if mat_path is None:
        print(f"  Sub {sid:02d}: no file found, skipping")
        continue
    try:
        X, y = preprocess_subject(mat_path)
    except Exception as exc:
        print(f"  Sub {sid:02d}: preprocess error — {exc}")
        continue

    fold_res = run_subject(sid, X, y, CONFIG, BASE_MODEL, DEVICE)
    ALL_FOLD_RESULTS.extend(fold_res)

    accs  = [r["accuracy"] for r in fold_res]
    baccs = [r["balanced_accuracy"] for r in fold_res]
    collapses = sum(1 for r in fold_res if r["collapse_flag"])
    mean_bacc = np.mean(baccs)

    SUBJECT_SUMMARIES.append({
        "subject_id":             sid,
        "mean_accuracy":          float(np.mean(accs)),
        "std_accuracy":           float(np.std(accs)),
        "mean_balanced_accuracy": float(mean_bacc),
        "std_balanced_accuracy":  float(np.std(baccs)),
        "n_folds":                len(fold_res),
        "n_collapsed_folds":      collapses,
    })

    print(f"  Sub {sid:02d}: bal_acc={mean_bacc*100:.1f}% ± {np.std(baccs)*100:.1f}%  "
          f"collapse={collapses}/{len(fold_res)}")

print("=" * 60)
print(f"Done. Total folds: {len(ALL_FOLD_RESULTS)}")

Running 50 subjects | clf=logistic_l2
  [hook=feature_encoder pool=mean] embedding matrix: (24, 64)


/home/vegorov/Repos/eeg_jepa_research/.venv/lib/python3.11/site-packages/sklearn/linear_model/_logistic.py:1135: FutureWarning: 'penalty' was deprecated in version 1.8 and will be removed in 1.10. To avoid this warning, leave 'penalty' set to its default value and use 'l1_ratio' or 'C' instead. Use l1_ratio=0 instead of penalty='l2', l1_ratio=1 instead of penalty='l1', and C=np.inf instead of penalty=None.
  warnings.warn(
/home/vegorov/Repos/eeg_jepa_research/.venv/lib/python3.11/site-packages/sklearn/linear_model/_logistic.py:1135: FutureWarning: 'penalty' was deprecated in version 1.8 and will be removed in 1.10. To avoid this warning, leave 'penalty' set to its default value and use 'l1_ratio' or 'C' instead. Use l1_ratio=0 instead of penalty='l2', l1_ratio=1 instead of penalty='l1', and C=np.inf instead of penalty=None.
  warnings.warn(
/home/vegorov/Repos/eeg_jepa_research/.venv/lib/python3.11/site-packages/sklearn/linear_model/_logistic.py:1135: FutureWarning: 'penalty' was depr

  Sub 01: bal_acc=43.1% ± 10.6%  collapse=0/10


/home/vegorov/Repos/eeg_jepa_research/.venv/lib/python3.11/site-packages/sklearn/linear_model/_logistic.py:1135: FutureWarning: 'penalty' was deprecated in version 1.8 and will be removed in 1.10. To avoid this warning, leave 'penalty' set to its default value and use 'l1_ratio' or 'C' instead. Use l1_ratio=0 instead of penalty='l2', l1_ratio=1 instead of penalty='l1', and C=np.inf instead of penalty=None.
  warnings.warn(
/home/vegorov/Repos/eeg_jepa_research/.venv/lib/python3.11/site-packages/sklearn/linear_model/_logistic.py:1135: FutureWarning: 'penalty' was deprecated in version 1.8 and will be removed in 1.10. To avoid this warning, leave 'penalty' set to its default value and use 'l1_ratio' or 'C' instead. Use l1_ratio=0 instead of penalty='l2', l1_ratio=1 instead of penalty='l1', and C=np.inf instead of penalty=None.
  warnings.warn(
/home/vegorov/Repos/eeg_jepa_research/.venv/lib/python3.11/site-packages/sklearn/linear_model/_logistic.py:1135: FutureWarning: 'penalty' was depr

  Sub 02: bal_acc=51.9% ± 11.9%  collapse=0/10


...

/home/vegorov/Repos/eeg_jepa_research/.venv/lib/python3.11/site-packages/sklearn/linear_model/_logistic.py:1135: FutureWarning: 'penalty' was deprecated in version 1.8 and will be removed in 1.10. To avoid this warning, leave 'penalty' set to its default value and use 'l1_ratio' or 'C' instead. Use l1_ratio=0 instead of penalty='l2', l1_ratio=1 instead of penalty='l1', and C=np.inf instead of penalty=None.
  warnings.warn(
/home/vegorov/Repos/eeg_jepa_research/.venv/lib/python3.11/site-packages/sklearn/linear_model/_logistic.py:1135: FutureWarning: 'penalty' was deprecated in version 1.8 and will be removed in 1.10. To avoid this warning, leave 'penalty' set to its default value and use 'l1_ratio' or 'C' instead. Use l1_ratio=0 instead of penalty='l2', l1_ratio=1 instead of penalty='l1', and C=np.inf instead of penalty=None.
  warnings.warn(
/home/vegorov/Repos/eeg_jepa_research/.venv/lib/python3.11/site-packages/sklearn/linear_model/_logistic.py:1135: FutureWarning: 'penalty' was depr

  Sub 49: bal_acc=42.5% ± 12.4%  collapse=0/10


/home/vegorov/Repos/eeg_jepa_research/.venv/lib/python3.11/site-packages/sklearn/linear_model/_logistic.py:1135: FutureWarning: 'penalty' was deprecated in version 1.8 and will be removed in 1.10. To avoid this warning, leave 'penalty' set to its default value and use 'l1_ratio' or 'C' instead. Use l1_ratio=0 instead of penalty='l2', l1_ratio=1 instead of penalty='l1', and C=np.inf instead of penalty=None.
  warnings.warn(
/home/vegorov/Repos/eeg_jepa_research/.venv/lib/python3.11/site-packages/sklearn/linear_model/_logistic.py:1135: FutureWarning: 'penalty' was deprecated in version 1.8 and will be removed in 1.10. To avoid this warning, leave 'penalty' set to its default value and use 'l1_ratio' or 'C' instead. Use l1_ratio=0 instead of penalty='l2', l1_ratio=1 instead of penalty='l1', and C=np.inf instead of penalty=None.
  warnings.warn(
/home/vegorov/Repos/eeg_jepa_research/.venv/lib/python3.11/site-packages/sklearn/linear_model/_logistic.py:1135: FutureWarning: 'penalty' was depr

  Sub 50: bal_acc=40.0% ± 10.2%  collapse=0/10
Done. Total folds: 500


/home/vegorov/Repos/eeg_jepa_research/.venv/lib/python3.11/site-packages/sklearn/linear_model/_logistic.py:1135: FutureWarning: 'penalty' was deprecated in version 1.8 and will be removed in 1.10. To avoid this warning, leave 'penalty' set to its default value and use 'l1_ratio' or 'C' instead. Use l1_ratio=0 instead of penalty='l2', l1_ratio=1 instead of penalty='l1', and C=np.inf instead of penalty=None.
  warnings.warn(


In [23]:
# --- Cache FROZEN rich embeddings per subject (for the TWFB hybrid, Branch A) ---
# Deterministic, label-free: uses the frozen pretrained BASE_MODEL (no per-fold fine-tune),
# so each trial maps to one fixed embedding the hybrid can load directly.
if CONFIG.get("save_embeddings", False):
    emb_dir = ARTIFACT_ROOT / "embeddings"
    emb_dir.mkdir(parents=True, exist_ok=True)
    frozen = copy.deepcopy(BASE_MODEL).to(DEVICE).eval()
    manifest = {"hook": CONFIG.get("embedding_hook", "feature_encoder"),
                "pool": CONFIG.get("embedding_pool", "mean"),
                "window_samples": WINDOW_SAMPLES, "sfreq_model": SFREQ_MODEL,
                "mi_window_s": list(CONFIG["mi_window_s"]), "subjects": []}
    n_saved = 0
    for sid in SUBJECT_IDS:
        mat_path = sid_to_path.get(sid)
        if mat_path is None:
            continue
        try:
            Xs, ys = preprocess_subject(mat_path)
            Es = extract_embeddings(frozen, Xs, CONFIG, DEVICE)   # (40, D), frozen
            np.savez(emb_dir / f"sub-{sid:02d}.npz", X=Es, y=ys)
            manifest["subjects"].append(int(sid)); manifest["embedding_dim"] = int(Es.shape[1])
            n_saved += 1
        except Exception as exc:
            print(f"  Sub {sid:02d}: embedding cache error — {exc}")
    json.dump(manifest, open(emb_dir / "manifest.json", "w"), indent=2)
    print(f"Cached frozen embeddings for {n_saved} subjects (D={manifest.get('embedding_dim')}) -> {emb_dir}")
else:
    print("save_embeddings=False -> skipping frozen embedding cache")

Cached frozen embeddings for 50 subjects (D=64) -> ../../artifacts/liu2024_sjepa_embeddings_lda/embeddings


## 10. Aggregate Results

In [24]:

if not ALL_FOLD_RESULTS:
    raise RuntimeError(
        "ALL_FOLD_RESULTS is empty — no folds completed successfully. "
        "Check that subjects loaded and that the run cell (Section 9) executed without errors."
    )

fold_df    = pd.DataFrame(ALL_FOLD_RESULTS)
subject_df = pd.DataFrame(SUBJECT_SUMMARIES)

all_baccs = fold_df["balanced_accuracy"].values
all_accs  = fold_df["accuracy"].values
n_collapsed = int(fold_df["collapse_flag"].sum())

cm_total = np.zeros((2, 2), dtype=int)
for row in ALL_FOLD_RESULTS:
    cm_total += np.array(row["confusion_matrix"])

left_recall_agg  = cm_total[0, 0] / cm_total[0].sum() if cm_total[0].sum() > 0 else float("nan")
right_recall_agg = cm_total[1, 1] / cm_total[1].sum() if cm_total[1].sum() > 0 else float("nan")

global_summary = {
    "method":                 f"sjepa_embeddings_{CONFIG['classifier']}",
    "classifier":             CONFIG["classifier"],
    "finetune_spatial_conv":  CONFIG["finetune_spatial_conv"],
    "n_subjects":             len(SUBJECT_SUMMARIES),
    "n_folds_total":          len(ALL_FOLD_RESULTS),
    "mean_accuracy":          float(np.mean(all_accs)),
    "std_accuracy":           float(np.std(all_accs)),
    "mean_balanced_accuracy": float(np.mean(all_baccs)),
    "std_balanced_accuracy":  float(np.std(all_baccs)),
    "n_collapsed_folds":      n_collapsed,
    "collapse_rate":          float(n_collapsed / max(len(ALL_FOLD_RESULTS), 1)),
    "left_recall_agg":        float(left_recall_agg),
    "right_recall_agg":       float(right_recall_agg),
    "confusion_matrix":       cm_total.tolist(),
}

print("=" * 60)
print(f"GLOBAL RESULTS — S-JEPA embeddings + {CONFIG['classifier']}")
print(f"  Mean Balanced Accuracy: {global_summary['mean_balanced_accuracy']*100:.2f}% ± {global_summary['std_balanced_accuracy']*100:.2f}%")
print(f"  Mean Accuracy:          {global_summary['mean_accuracy']*100:.2f}% ± {global_summary['std_accuracy']*100:.2f}%")
print(f"  Left recall:  {left_recall_agg*100:.1f}%  |  Right recall: {right_recall_agg*100:.1f}%")
print(f"  Collapsed folds: {n_collapsed}/{len(ALL_FOLD_RESULTS)} ({global_summary['collapse_rate']*100:.1f}%)")
print(f"  Aggregated CM: {cm_total.tolist()}")
print("=" * 60)


GLOBAL RESULTS — S-JEPA embeddings + logistic_l2
  Mean Balanced Accuracy: 44.31% ± 12.17%
  Mean Accuracy:          44.31% ± 12.17%
  Left recall:  44.6%  |  Right recall: 44.0%
  Collapsed folds: 0/500 (0.0%)
  Aggregated CM: [[1783, 2217], [2238, 1762]]


## 11. Save Artifacts

In [25]:
fold_df.to_csv(ARTIFACT_ROOT / "fold_results.csv", index=False)
subject_df.to_csv(ARTIFACT_ROOT / "subject_summary.csv", index=False)
with open(ARTIFACT_ROOT / "global_summary.json", "w") as f:
    json.dump(global_summary, f, indent=2)
with open(ARTIFACT_ROOT / "config.json", "w") as f:
    json.dump(CONFIG, f, indent=2, default=str)

print(f"Saved: fold_results.csv, subject_summary.csv, global_summary.json, config.json")

Saved: fold_results.csv, subject_summary.csv, global_summary.json, config.json


## 12. Plots

In [26]:
# Confusion matrix
fig, ax = plt.subplots(figsize=(4, 3))
im = ax.imshow(cm_total, cmap="Blues")
ax.set_xticks([0, 1]); ax.set_xticklabels(["Pred Left", "Pred Right"])
ax.set_yticks([0, 1]); ax.set_yticklabels(["True Left", "True Right"])
for i in range(2):
    for j in range(2):
        ax.text(j, i, str(cm_total[i, j]), ha="center", va="center", fontsize=12)
ax.set_title(f"Aggregated CM — S-JEPA + {CONFIG['classifier']}")
plt.colorbar(im, ax=ax)
plt.tight_layout()
plt.savefig(ARTIFACT_ROOT / "confusion_matrix.png", dpi=150, bbox_inches="tight")
plt.show()

# Per-subject accuracy
fig, ax = plt.subplots(figsize=(max(8, len(subject_df) * 0.4), 4))
sids  = subject_df["subject_id"].values
baccs = subject_df["mean_balanced_accuracy"].values * 100
stds  = subject_df["std_balanced_accuracy"].values * 100
ax.bar(sids, baccs, yerr=stds, capsize=3, color="mediumpurple", alpha=0.8)
ax.axhline(50, color="red", linestyle="--", label="Chance")
ax.axhline(float(np.mean(baccs)), color="orange", linestyle="-", label=f"Mean={np.mean(baccs):.1f}%")
ax.set_xlabel("Subject ID")
ax.set_ylabel("Balanced Accuracy (%)")
ax.set_title(f"Per-subject — S-JEPA + {CONFIG['classifier']}")
ax.legend()
ax.set_xticks(sids)
ax.set_xticklabels(sids, rotation=90, fontsize=7)
plt.tight_layout()
plt.savefig(ARTIFACT_ROOT / "subject_accuracy_plot.png", dpi=150, bbox_inches="tight")
plt.show()
print("Plots saved.")

/tmp/ipykernel_1619760/505937107.py:13: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


Plots saved.


/tmp/ipykernel_1619760/505937107.py:31: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()
